# YOLO11s Fine-Tuning -- Marine Object Detection (partial / backbone-frozen)

**Dataset**: `ssriram2005/marine-dataset` (YOLO format, 7 classes: boat, buoy, cargo, ferry, tug, warship, yacht)  
**Base weights**: YOLO11s, pretrained on COCO  
**Fine-tuning strategy**: freeze the backbone's early layers (low-level color/edge/shape filters) and only train from the mid backbone onward + full head. Rationale: these are real-world RGB images of vessels/boats -- visually much closer to COCO's natural-image distribution than something like SAR imagery, so we don't need SARDet-100K-style full fine-tuning. Early filters COCO already learned (edges, color blobs, simple textures) should transfer directly.

### Why YOLO11s
- YOLO11 (Ultralytics, 2024) is the current generation -- better accuracy/speed tradeoff than YOLOv8 at similar size, and still has the same simple `model.train(freeze=N)` partial-fine-tune API used below.
- `-s` (small) size: dataset is ~2.5K images across 7 classes -- not enough data to justify `-m`/`-l`'s extra capacity, and `-n` (nano) risks underfitting the fine-grained cargo/ferry/warship distinctions. `-s` is the standard sweet spot for datasets this size.

### How to run
1. Attach `ssriram2005/marine-dataset` as the input dataset
2. Accelerator: GPU T4 x2 (single GPU is used; see note in training cell)
3. Save and Run All

## 1 - Install dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'seaborn==0.12.2',
    'ultralytics>=8.3.0',
])
print('Done')

## 2 - Imports, seeds, GPU check

In [ ]:
import os, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import cv2

import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), 'No GPU -- set Accelerator to GPU T4 x2'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' GPU', i, ':', torch.cuda.get_device_name(i))

## 3 - Locate dataset and fix data.yaml paths

In [ ]:
WORK_DIR = Path('/kaggle/working')
RESULTS_DIR = WORK_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = Path('/kaggle/input/datasets/ssriram2005/marine-detection-balanced/Clean_YOLO_Marine_Dataset')
assert DATASET_ROOT.exists(), f'Dataset not found at {DATASET_ROOT} -- check attached dataset path'

with open(DATASET_ROOT / 'data.yaml') as f:
    data_yaml = yaml.safe_load(f)

CLASS_NAMES = data_yaml['names']
print('Classes:', CLASS_NAMES)

for split in ['train', 'valid', 'test']:
    n_img = len(list((DATASET_ROOT / split / 'images').glob('*.*')))
    n_lbl = len(list((DATASET_ROOT / split / 'labels').glob('*.txt')))
    print(f'{split}: {n_img} images, {n_lbl} labels')

# Kaggle's /kaggle/input is read-only, and Roboflow's data.yaml uses relative
# paths (../train/images) that assume a specific working directory -- write a
# fixed copy with absolute paths into /kaggle/working instead.
fixed_yaml = {
    'train': str(DATASET_ROOT / 'train' / 'images'),
    'val':   str(DATASET_ROOT / 'valid' / 'images'),
    'test':  str(DATASET_ROOT / 'test'  / 'images'),
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

DATA_YAML_PATH = WORK_DIR / 'marine_data.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(fixed_yaml, f)

print('\nWrote fixed data.yaml to', DATA_YAML_PATH)
print(yaml.safe_dump(fixed_yaml))

## 4 - Load pretrained YOLO11s and inspect layer indices

Ultralytics' `freeze=N` argument freezes the first `N` modules of `model.model` (the sequential layer list printed below). YOLO11's backbone is roughly layers `0-9` (stem through SPPF), with the neck (layers `10-22`ish) and detection head after that -- exact boundary printed below so we pick `N` correctly instead of guessing.

In [ ]:
base_model = YOLO('yolo11s.pt')

print(f"{'idx':>4}  {'module type':<30} params")
for i, m in enumerate(base_model.model.model):
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{i:>4}  {m.__class__.__name__:<30} {n_params:,}')

## 5 - Fine-tune: freeze early backbone layers, train the rest

`FREEZE_LAYERS = 4` freezes the stem + first couple of backbone conv/C3k2 blocks (low-level color/edge/texture filters) and leaves everything from the mid-backbone through the detection head trainable -- i.e. exactly the "retain early layers, fine-tune middle-to-end" strategy. Adjust this number after looking at the layer printout above if you want to freeze more/less.

This is a **single YOLO `.train()` call** -- Ultralytics handles the freeze internally (zeroes gradients for the frozen modules), no separate architecture surgery needed.

In [ ]:
FREEZE_LAYERS = 4   # freeze stem + first backbone stage(s); tune based on layer printout above
IMGSZ = 640
EPOCHS = 80
BATCH = 16

model = YOLO('yolo11s.pt')

results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    freeze=FREEZE_LAYERS,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,          # lower LR than a full-from-scratch fine-tune, since most of the network is pretrained/frozen
    cos_lr=True,
    seed=SEED,
    project=str(WORK_DIR / 'runs'),
    name='yolo11s_marine_partial_ft',
    device=0,            # single GPU -- see note below for multi-GPU
    plots=True,
    verbose=True,
)

print('Training complete. Best weights at:', model.trainer.best)

**Multi-GPU note:** to actually use both T4s, pass `device=[0,1]` and increase `batch` accordingly (e.g. `batch=32`). Left as single-GPU (`device=0`) above since ~2.5K images trains quickly enough on one T4 and avoids DDP overhead/setup issues for a first run -- switch it on if you want faster wall-clock time.

## 6 - Training curves

In [ ]:
run_dir = Path(model.trainer.save_dir)
results_csv = run_dir / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = [c.strip() for c in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df['epoch'], df['train/box_loss'], label='train box loss')
axes[0].plot(df['epoch'], df['val/box_loss'], label='val box loss')
axes[0].plot(df['epoch'], df['train/cls_loss'], label='train cls loss')
axes[0].plot(df['epoch'], df['val/cls_loss'], label='val cls loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('Training / validation loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50')
axes[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mAP')
axes[1].set_title('Validation mAP'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 - Evaluate best checkpoint on the held-out test split

In [ ]:
best_weights = run_dir / 'weights' / 'best.pt'
model_best = YOLO(str(best_weights))

test_metrics = model_best.val(
    data=str(DATA_YAML_PATH),
    split='test',
    imgsz=IMGSZ,
    device=0,
    plots=True,
    save_json=True,
    project=str(WORK_DIR / 'runs'),
    name='yolo11s_marine_test_eval',
)

print('\nOverall test metrics')
print('mAP50    :', test_metrics.box.map50)
print('mAP50-95 :', test_metrics.box.map)
print('Precision:', test_metrics.box.mp)
print('Recall   :', test_metrics.box.mr)

## 8 - Per-class AP bar chart

In [ ]:
ap50 = test_metrics.box.ap50
ap5095 = test_metrics.box.ap.mean(axis=1) if test_metrics.box.ap.ndim > 1 else test_metrics.box.ap

x = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, ap50, width, label='AP@50', color='#4C72B0')
bars2 = ax.bar(x + width/2, ap5095, width, label='AP@50-95', color='#DD8452')

for bar, v in zip(bars1, ap50):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.012, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
for bar, v in zip(bars2, ap5095):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.012, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=10)
ax.set_ylabel('Average Precision'); ax.set_ylim(0, 1.12)
ax.set_title('Per-class AP -- test set (YOLO11s, partial fine-tune)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_ap.png', dpi=150, bbox_inches='tight')
plt.show()

pc_df = pd.DataFrame({'Class': CLASS_NAMES, 'AP@50': ap50, 'AP@50-95': ap5095})
print(pc_df.to_string(index=False))

## 9 - Ground truth vs prediction panels (10 test images)

In [ ]:
gc.collect(); torch.cuda.empty_cache()

TEST_IMG_DIR = DATASET_ROOT / 'test' / 'images'
TEST_LBL_DIR = DATASET_ROOT / 'test' / 'labels'

NUM_SAMPLES = 10
test_imgs = list(TEST_IMG_DIR.glob('*.*'))
selected = random.sample(test_imgs, min(NUM_SAMPLES, len(test_imgs)))

COLOR_GT, COLOR_PRED = (0, 200, 80), (0, 180, 255)
FONT = cv2.FONT_HERSHEY_SIMPLEX

fig, axes = plt.subplots(NUM_SAMPLES, 2, figsize=(14, 5 * NUM_SAMPLES))
fig.suptitle('Ground Truth (left) vs YOLO11s Partial Fine-Tune Predictions (right) -- conf >= 0.25',
             fontsize=14, fontweight='bold', y=1.001)

for i, img_path in enumerate(selected):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    img_gt = img_rgb.copy()
    lbl_path = TEST_LBL_DIR / (img_path.stem + '.txt')
    gt_count = 0
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) != 5: continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1, y1 = int((cx - bw/2)*w), int((cy - bh/2)*h)
            x2, y2 = int((cx + bw/2)*w), int((cy + bh/2)*h)
            cv2.rectangle(img_gt, (x1, y1), (x2, y2), COLOR_GT, 2)
            label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            (tw, th), _ = cv2.getTextSize(label, FONT, 0.45, 1)
            cv2.rectangle(img_gt, (x1, max(y1-th-4, 0)), (x1+tw+2, y1), COLOR_GT, -1)
            cv2.putText(img_gt, label, (x1+1, max(y1-3, th)), FONT, 0.45, (0,0,0), 1, cv2.LINE_AA)
            gt_count += 1

    img_pred = img_rgb.copy()
    res = model_best.predict(str(img_path), imgsz=IMGSZ, conf=0.25, device=0, verbose=False)[0]
    pred_count = 0
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls[0]); conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cv2.rectangle(img_pred, (x1, y1), (x2, y2), COLOR_PRED, 2)
            label = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(label, FONT, 0.45, 1)
            cv2.rectangle(img_pred, (x1, max(y1-th-4, 0)), (x1+tw+2, y1), COLOR_PRED, -1)
            cv2.putText(img_pred, label, (x1+1, max(y1-3, th)), FONT, 0.45, (0,0,0), 1, cv2.LINE_AA)
            pred_count += 1

    axes[i, 0].imshow(img_gt); axes[i, 0].set_title(f'GT -- {img_path.name} ({gt_count} objects)', fontsize=9); axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pred); axes[i, 1].set_title(f'Predicted -- {pred_count} detections', fontsize=9); axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'gt_vs_pred_panels.png', dpi=120, bbox_inches='tight')
plt.show()

## 10 - Save outputs summary

In [ ]:
import shutil
shutil.copy(best_weights, RESULTS_DIR / 'best.pt')
shutil.copy(DATA_YAML_PATH, RESULTS_DIR / 'marine_data.yaml')

print('All saved outputs under /kaggle/working/results/:')
print('{:<40} {:>8}'.format('File', 'Size'))
print('-' * 50)
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        unit = 'MB' if sz > 1_000_000 else 'KB'
        val = sz / 1_000_000 if sz > 1_000_000 else sz / 1024
        print('{:<40} {:>6.1f} {}'.format(str(f.relative_to(RESULTS_DIR)), val, unit))